In [10]:
import pandas as pd
import numpy as np

In [11]:
# Load listings
listings = pd.read_csv("../data/processed/london_listings_engineered.csv")

In [13]:
# Load calendar
import pandas as pd

calendar_iter = pd.read_csv(
    "../data/raw/calendar.csv",
    usecols=["listing_id", "date", "available", "price"],
    chunksize=500000
)

calendar = next(calendar_iter)
calendar.head()

,listing_id,date,available,price
0,1196288722069341420,2025-09-15,f,NaN
1,1196288722069341420,2025-09-16,f,NaN
2,1196288722069341420,2025-09-17,f,NaN
3,1196288722069341420,2025-09-18,f,NaN
4,1196288722069341420,2025-09-19,f,NaN


In [5]:
import pandas as pd
import numpy as np

In [8]:
chunks = pd.read_csv(
    "../data/raw/calendar.csv",
    usecols=["listing_id", "available"],
    chunksize=300000
)

availability_list = []

for chunk in chunks:

    chunk["available"] = chunk["available"].map({"t": 1, "f": 0})

    agg = chunk.groupby("listing_id").agg(
        availability_rate=("available", "mean")
    ).reset_index()

    availability_list.append(agg)

calendar_agg = pd.concat(availability_list)

calendar_agg = (
    calendar_agg
    .groupby("listing_id")
    .mean()
    .reset_index()
)

calendar_agg.head()

,listing_id,availability_rate
0,13913,0.906849
1,15400,0.545205
2,17402,0.219178
3,24328,0.805479
4,36274,0.884932


In [12]:
import pandas as pd

listings = pd.read_csv(
    "../data/processed/london_listings_engineered.csv"
)

print(listings.shape)
listings.head()

(96871, 76)


,id,listing_url,scrape_id,last_scraped,source,name,description,neighborhood_overview,picture_url,host_id,...,review_scores_checkin,review_scores_communication,review_scores_location,review_scores_value,instant_bookable,calculated_host_listings_count,calculated_host_listings_count_entire_homes,calculated_host_listings_count_private_rooms,calculated_host_listings_count_shared_rooms,reviews_per_month
0,13913,https://www.airbnb.com/rooms/13913,20250914034649,2025-09-16,city scrape,Holiday London DB Room Let-on going,My bright double bedroom with a large window h...,Finsbury Park is a friendly melting pot commun...,https://a0.muscache.com/pictures/miso/Hosting-...,54730,...,4.81,4.87,4.78,4.78,False,2,1,1,0,0.30
1,15400,https://www.airbnb.com/rooms/15400,20250914034649,2025-09-16,city scrape,Bright Chelsea Apartment. Chelsea!,Lots of windows and light. St Luke's Gardens ...,It is Chelsea.,https://a0.muscache.com/pictures/428392/462d26...,60302,...,4.88,4.84,4.93,4.74,False,1,1,0,0,0.51
2,17402,https://www.airbnb.com/rooms/17402,20250914034649,2025-09-16,city scrape,Very Central Modern 3-Bed/2 Bath By Oxford St W1,"You'll have a great time in this beautiful, cl...","Fitzrovia is a very desirable trendy, arty and...",https://a0.muscache.com/pictures/39d5309d-fba7...,67564,...,4.72,4.72,4.89,4.61,False,2,2,0,0,0.32
3,24328,https://www.airbnb.com/rooms/24328,20250914034649,2025-09-18,previous scrape,Battersea live/work artist house,"Artist house by SW Battersea Park, bright high...","- Battersea is a quiet family area, easy acces...",https://a0.muscache.com/pictures/9194b40f-c627...,41759,...,4.90,4.93,4.60,4.65,False,1,1,0,0,0.53
4,36274,https://www.airbnb.com/rooms/36274,20250914034649,2025-09-15,city scrape,Bright 1 bedroom apt off brick lane in Shoreditch,*Update June '25- Pump Installed to improve wa...,NaN,https://a0.muscache.com/pictures/hosting/Hosti...,133271,...,4.62,4.46,4.85,4.54,True,2,2,0,0,0.09


In [13]:
master_df = listings.merge(
    calendar_agg,
    left_on="id",
    right_on="listing_id",
    how="left"
)

print(master_df.shape)

master_df.head()

(96871, 78)


,id,listing_url,scrape_id,last_scraped,source,name,description,neighborhood_overview,picture_url,host_id,...,review_scores_location,review_scores_value,instant_bookable,calculated_host_listings_count,calculated_host_listings_count_entire_homes,calculated_host_listings_count_private_rooms,calculated_host_listings_count_shared_rooms,reviews_per_month,listing_id,availability_rate
0,13913,https://www.airbnb.com/rooms/13913,20250914034649,2025-09-16,city scrape,Holiday London DB Room Let-on going,My bright double bedroom with a large window h...,Finsbury Park is a friendly melting pot commun...,https://a0.muscache.com/pictures/miso/Hosting-...,54730,...,4.78,4.78,False,2,1,1,0,0.30,13913,0.906849
1,15400,https://www.airbnb.com/rooms/15400,20250914034649,2025-09-16,city scrape,Bright Chelsea Apartment. Chelsea!,Lots of windows and light. St Luke's Gardens ...,It is Chelsea.,https://a0.muscache.com/pictures/428392/462d26...,60302,...,4.93,4.74,False,1,1,0,0,0.51,15400,0.545205
2,17402,https://www.airbnb.com/rooms/17402,20250914034649,2025-09-16,city scrape,Very Central Modern 3-Bed/2 Bath By Oxford St W1,"You'll have a great time in this beautiful, cl...","Fitzrovia is a very desirable trendy, arty and...",https://a0.muscache.com/pictures/39d5309d-fba7...,67564,...,4.89,4.61,False,2,2,0,0,0.32,17402,0.219178
3,24328,https://www.airbnb.com/rooms/24328,20250914034649,2025-09-18,previous scrape,Battersea live/work artist house,"Artist house by SW Battersea Park, bright high...","- Battersea is a quiet family area, easy acces...",https://a0.muscache.com/pictures/9194b40f-c627...,41759,...,4.60,4.65,False,1,1,0,0,0.53,24328,0.805479
4,36274,https://www.airbnb.com/rooms/36274,20250914034649,2025-09-15,city scrape,Bright 1 bedroom apt off brick lane in Shoreditch,*Update June '25- Pump Installed to improve wa...,NaN,https://a0.muscache.com/pictures/hosting/Hosti...,133271,...,4.85,4.54,True,2,2,0,0,0.09,36274,0.884932


In [14]:
master_df.drop(columns=["listing_id"], inplace=True)

In [15]:
master_df[["id", "availability_rate"]].head()

,id,availability_rate
0,13913,0.906849
1,15400,0.545205
2,17402,0.219178
3,24328,0.805479
4,36274,0.884932


In [20]:
master_df.to_csv(
    "../data/processed/london_master_dataset.csv",
    index=False
)

print("Master dataset created successfully!")

Master dataset created successfully!
